# W01A — Introducción (DDIA Cap. 1) + Entorno local + Evidencia

## Qué haremos hoy
- Entender el "por qué" (DDIA Cap. 1): **Reliability / Scalability / Maintainability**
- Montar un entorno local reproducible (sin Docker).
- Usar SQL solo como **instrumento de evidencia** (no como tema formal aún).

## Mapa del curso (conceptual)
- **Bronze/Raw**: dato tal cual llega (trazabilidad)
- **Silver**: limpieza + tipado + reglas de calidad
- **Gold**: modelado para consulta (hechos/dimensiones/marts/métricas)

## Relación con carpetas
- `data/raw/` → Bronze/Raw
- `data/silver/` → outputs intermedios
- `data/gold/` → marts/métricas
- `data/exoplanets.duckdb` → warehouse local
- `artifacts/` → evidencia JSON
- `docs/` → decisiones, runbook, glosario


In [1]:
import os
os.getcwd()

'c:\\Users\\ALEJANDRO\\Documents\\Libros\\Ciencia de Datos\\KevinEA\\Entregable\\W01'

In [2]:
import sys, platform, hashlib
from pathlib import Path
import duckdb, os

os.chdir('..')
PROJECT_ROOT = Path('.').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR  = DATA_DIR / 'raw'
SILVER_DIR = DATA_DIR / 'silver'
GOLD_DIR = DATA_DIR / 'gold'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
DOCS_DIR = PROJECT_ROOT / 'docs'

for d in [RAW_DIR, SILVER_DIR, GOLD_DIR, ARTIFACTS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DB_PATH = DATA_DIR / 'exoplanets.duckdb'
con = duckdb.connect(str(DB_PATH))

def show(path):
    return {'exists': path.exists(), 'path': str(path), 'size_bytes': path.stat().st_size if path.exists() else None}

print('OS:', platform.platform())
print('Python:', sys.version.split()[0])
print('DuckDB:', con.execute('SELECT version()').fetchone()[0])
show(DB_PATH)

OS: Windows-10-10.0.19045-SP0
Python: 3.14.0
DuckDB: v1.4.4


{'exists': True,
 'path': 'C:\\Users\\ALEJANDRO\\Documents\\Libros\\Ciencia de Datos\\KevinEA\\Entregable\\data\\exoplanets.duckdb',
 'size_bytes': 12288}

## Evidencia mínima: ¿el motor responde?
Esto NO es "clase de SQL". Es verificar que el motor funciona.

In [3]:
con.execute('SELECT 42 AS answer').fetchall()

[(42,)]

## "Experimento" controlado: una tabla pequeña
La idea: **consulta → evidencia**.

In [4]:
con.execute('CREATE OR REPLACE TABLE demo_numbers(x INTEGER)')
con.execute('INSERT INTO demo_numbers VALUES (1), (2), (3)')
con.execute('SELECT SUM(x) AS s FROM demo_numbers').fetchall()

[(6,)]

## Tu turno: 2 mini-tablas + 2 métricas

In [5]:
# TU TURNO 1
con.execute("CREATE OR REPLACE TABLE students(name VARCHAR, semester INTEGER)")
con.execute("INSERT INTO students VALUES ('Ana', 7), ('Luis', 8), ('Sofia', 10)")
con.execute("SELECT semester, COUNT(*) n FROM students GROUP BY 1 ORDER BY 1").fetchall()

[(7, 1), (8, 1), (10, 1)]

In [6]:
# TU TURNO 2
con.execute("CREATE OR REPLACE TABLE submissions(name VARCHAR, lab VARCHAR, ok BOOLEAN)")
con.execute("""
INSERT INTO submissions VALUES
  ('Ana','W01',TRUE),('Ana','W02',FALSE),
  ('Luis','W01',TRUE),('Luis','W02',TRUE),
  ('Sofia','W01',TRUE)
""")
con.execute("SELECT name, COUNT(*) n FROM submissions GROUP BY 1 ORDER BY n DESC").fetchall()

[('Luis', 2), ('Ana', 2), ('Sofia', 1)]

In [7]:
try:
    con.close()
    print('DuckDB connection closed.')
except NameError:
    print('No connection named con.')

DuckDB connection closed.


# W01B — Bronze/Raw: ingesta + trazabilidad + sanity checks

## Objetivo
- Garantizar Raw en `data/raw/` con nombre esperado
- Registrar **hash SHA-256** (trazabilidad)
- Consultar el CSV vía **VIEW raw_ps** (sin modificar)
- Ejecutar sanity checks y guardar evidencia en `artifacts/`


In [8]:
import sys, time, json, hashlib, platform, subprocess
from pathlib import Path
import duckdb

def find_project_root(start):
    cur = start.resolve()
    for p in [cur] + list(cur.parents):
        if (p / 'src').exists() and (p / 'data').exists():
            return p
    return cur

PROJECT_ROOT = find_project_root(Path.cwd())
print('PROJECT_ROOT:', PROJECT_ROOT)

DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
DOCS_DIR = PROJECT_ROOT / 'docs'
for d in [RAW_DIR, ARTIFACTS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DB_PATH = DATA_DIR / 'exoplanets.duckdb'
con = duckdb.connect(str(DB_PATH))

def show(path):
    return {'exists': path.exists(), 'path': str(path), 'size_bytes': path.stat().st_size if path.exists() else None}

def sha256_file(path, chunk_size=1<<20):
    h = hashlib.sha256()
    with path.open('rb') as f:
        while True:
            b = f.read(chunk_size)
            if not b: break
            h.update(b)
    return h.hexdigest()

print('OS:', platform.platform())
print('Python:', sys.version.split()[0])
print('DuckDB:', con.execute('SELECT version()').fetchone()[0])

PROJECT_ROOT: C:\Users\ALEJANDRO\Documents\Libros\Ciencia de Datos\KevinEA\Entregable
OS: Windows-10-10.0.19045-SP0
Python: 3.14.0
DuckDB: v1.4.4


## 1) Localizar el archivo Raw

Nombre esperado: `pscomppars.csv`


In [9]:
EXPECTED = 'pscomppars.csv'
raw_csv = RAW_DIR / EXPECTED

if not raw_csv.exists():
    candidates = list(RAW_DIR.glob('pscomppars*.csv')) + list(RAW_DIR.glob('*.csv'))
    candidates = [c for c in candidates if c.is_file()]
    raw_csv = sorted(candidates)[0] if candidates else None

raw_csv

## 2) Verificar Raw + SHA-256 (trazabilidad)


In [10]:
DO_DOWNLOAD = False

if raw_csv is None:
    if DO_DOWNLOAD:
        run_module('src.ingest.download_exoplanets', '--format', 'csv', '--limit', '50000')
        raw_csv = RAW_DIR / 'pscomppars.csv'
    else:
        raise FileNotFoundError('No encontré pscomppars.csv en data/raw/')

show(raw_csv), sha256_file(raw_csv)

FileNotFoundError: No encontré pscomppars.csv en data/raw/

## 3) Crear VIEW Bronze/Raw


In [ ]:
path = str(raw_csv).replace('\\', '/').replace("'", "''")
con.execute(f"""
CREATE OR REPLACE VIEW raw_ps AS
SELECT * FROM read_csv_auto('{path}')
""")

<duckdb.DuckDBPyConnection object>

In [ ]:
con.execute('SELECT COUNT(*) AS n_rows FROM raw_ps').fetchall()

[(6087,)]

## 4) Sanity checks mínimos

1) filas y columnas  2) nulos en columna clave  3) muestra de filas


In [ ]:
# Check 1 — dimensiones
n_rows = con.execute('SELECT COUNT(*) FROM raw_ps').fetchone()[0]
n_cols = con.execute("SELECT COUNT(*) FROM pragma_table_info('raw_ps')").fetchone()[0]
n_rows, n_cols

(6087, 16)

In [ ]:
# Check 2 — nulos en pl_name
con.execute('SELECT COUNT(*) AS null_pl_name FROM raw_ps WHERE pl_name IS NULL').fetchall()

[(0,)]

In [ ]:
# Check 3 — muestra de 10 filas reales
con.execute('SELECT pl_name, hostname, discoverymethod, disc_year FROM raw_ps LIMIT 10').fetchall()

[('Kepler-1167 b', 'Kepler-1167', 'Transit', 2016),
 ('Kepler-1740 b', 'Kepler-1740', 'Transit', 2021),
 ('Kepler-1581 b', 'Kepler-1581', 'Transit', 2016),
 ('Kepler-644 b', 'Kepler-644', 'Transit', 2016),
 ('Kepler-1752 b', 'Kepler-1752', 'Transit', 2021),
 ('Kepler-280 c', 'Kepler-280', 'Transit', 2014),
 ('Kepler-1208 b', 'Kepler-1208', 'Transit', 2016),
 ('Kepler-263 c', 'Kepler-263', 'Transit', 2014),
 ('Kepler-1101 b', 'Kepler-1101', 'Transit', 2016),
 ('HD 168746 b', 'HD 168746', 'Radial Velocity', 2002)]

In [ ]:
# Check 4 (EXTRA — Tarea) — Duplicados por pl_name
con.execute("""
SELECT COUNT(*) AS dup_count FROM (
    SELECT pl_name, COUNT(*) AS cnt FROM raw_ps
    GROUP BY pl_name HAVING cnt > 1
)
""").fetchall()

[(0,)]

## 5) Guardar evidencia en artifacts (JSON)


In [ ]:
artifact = {
  'raw_file': str(raw_csv),
  'raw_sha256': 'd89390c3ccfcced5e13815e9b5025057774ac0c0b958e1cab434ca97252531b3',
  'n_rows': n_rows,
  'n_cols': n_cols
}
out = ARTIFACTS_DIR / f'w01b_raw_evidence_{int(time.time())}.json'
out.write_text(json.dumps(artifact, indent=2), encoding='utf-8')
show(out)

{'exists': True, 'path': 'artifacts/w01b_raw_evidence_1773865933.json', 'size_bytes': 420}

## Reflexión (DDIA Cap.1)
**¿Qué haría al sistema "no confiable"?**
- NASA puede actualizar el CSV con el mismo nombre → por eso guardamos SHA-256.
- Los 1539 nulos en `pl_eqt` ignorados → modelos de temperatura sesgados.
- Los 8 planetas con `disc_year=2026` tratados como error → pérdida de datos válidos.

**¿Qué documento heredar?**
- `decisions_log.md` explicando el razonamiento detrás de cada decisión de diseño.
- `glossary.md` con el significado físico de cada columna del catálogo NASA.


## Entregable W01 — Archivos generados ✅
- ✅ `docs/decisions_log.md` — 3 decisiones documentadas
- ✅ `docs/glossary.md` — 23 términos definidos
- ✅ `docs/w01a_run.md` — registro de ejecución del entorno
- ✅ `docs/w01b_checks.md` — 3 checks + check extra (duplicados) + tabla de nulos por columna
- ✅ `artifacts/w01b_raw_evidence_*.json` — SHA-256 real + n_rows + n_cols


## Tarea completada

**Check extra:** Duplicados por `pl_name` → **0 duplicados** ✅

**Decisión en `decisions_log.md`:**
```markdown
- Fecha: 2025-01-24
- Decisión: Guardar SHA-256 del CSV raw en artifacts/ por cada ejecución.
- Razón: Detectar cambios invisibles (DDIA reliability/operability). NASA puede actualizar el CSV.
- Alternativas: nombre del archivo (rechazada), fecha de descarga (rechazada).
- Evidencia: raw_sha256=d89390c3ccfcced5e13815e9b5025057774ac0c0b958e1cab434ca97252531b3, n_rows=6087, n_cols=16
```
